Purpose: Clean, standardize, harmonize, and consolidate Stores and Orders datasets from Company A and Company B into Silver Delta tables.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [0]:
base_path = "/Volumes/workspace/default/fmcg_data/"
bronze_path = base_path + "bronze/"
silver_path= "/Volumes/workspace/default/fmcg_data/silver/"

In [0]:
stores_a = spark.read.format("delta").load(bronze_path + "stores_a")
stores_b = spark.read.format("delta").load(bronze_path + "stores_b")



CLEANING STORES_A

In [0]:
stores_a.printSchema()
stores_b.printSchema()


root
 |-- Store_ID: string (nullable = true)
 |-- Store_Name: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Manager: string (nullable = true)
 |-- Opening_Date: date (nullable = true)
 |-- Created_At: timestamp (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)

root
 |-- Store_Code: string (nullable = true)
 |-- Branch_Name: string (nullable = true)
 |-- Town: string (nullable = true)
 |-- Province: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- Store_Manager: string (nullable = true)
 |-- Opening_Date: date (nullable = true)
 |-- Created_On: timestamp (nullable = true)
 |-- Updated_On: timestamp (nullable = true)



In [0]:
print("Rows:", stores_a.count())

Rows: 26


In [0]:
print("Missing values:\n")
stores_a.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in stores_a.columns
]).show()

Missing values:

+--------+----------+----+-----+------+-------+------------+----------+------------+
|Store_ID|Store_Name|City|State|Region|Manager|Opening_Date|Created_At|Last_Updated|
+--------+----------+----+-----+------+-------+------------+----------+------------+
|       0|         0|   0|    0|     0|      2|           0|         0|           0|
+--------+----------+----+-----+------+-------+------------+----------+------------+



In [0]:
stores_a = stores_a.dropDuplicates(["Store_ID"])

In [0]:
#standardise

stores_a = stores_a.withColumn(
    "City",
    initcap(col("City"))
)
stores_a = stores_a.withColumn(
    "State",
    initcap(col("State"))
)
stores_a = stores_a.withColumn(
    "Region",
    initcap(col("Region"))
)
stores_a = stores_a.withColumn(
    "Manager",
    initcap(col("Manager"))
)

In [0]:
# if manager is null , replacing by unknown
stores_a = stores_a.withColumn(
    "Manager",
    coalesce(col("Manager"), lit("Unknown"))
)

In [0]:
print("Rows:", stores_a.count())

stores_a.groupBy("Store_ID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

Rows: 25
+--------+-----+
|Store_ID|count|
+--------+-----+
+--------+-----+



CLEANING STORES_B

In [0]:
stores_b.printSchema()

#display(stores_b)

print("Rows:", stores_b.count())

root
 |-- Store_Code: string (nullable = true)
 |-- Branch_Name: string (nullable = true)
 |-- Town: string (nullable = true)
 |-- Province: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- Store_Manager: string (nullable = true)
 |-- Opening_Date: date (nullable = true)
 |-- Created_On: timestamp (nullable = true)
 |-- Updated_On: timestamp (nullable = true)

Rows: 20


In [0]:
print("Missing values:\n")
stores_b.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in stores_b.columns
]).show()

Missing values:

+----------+-----------+----+--------+----+-------------+------------+----------+----------+
|Store_Code|Branch_Name|Town|Province|Zone|Store_Manager|Opening_Date|Created_On|Updated_On|
+----------+-----------+----+--------+----+-------------+------------+----------+----------+
|         0|          0|   0|       0|   0|            2|           0|         0|         0|
+----------+-----------+----+--------+----+-------------+------------+----------+----------+



In [0]:
stores_b = stores_b.dropDuplicates(["Store_Code"])

In [0]:
stores_b = stores_b \
    .withColumn("Town", initcap(col("Town"))) \
    .withColumn("Province", initcap(col("Province"))) \
    .withColumn("Zone", initcap(col("Zone"))) \
    .withColumn("Store_Manager", initcap(col("Store_Manager")))

In [0]:
print("Rows:", stores_b.count())

stores_b.groupBy("Store_Code") \
    .count() \
    .filter(col("count") > 1) \
    .show()

#display(stores_b)

Rows: 18
+----------+-----+
|Store_Code|count|
+----------+-----+
+----------+-----+



In [0]:
stores_b = stores_b.fillna({"Store_Manager": "Unknown"})

Schema harmonization and merging


In [0]:
stores_b = stores_b \
    .withColumnRenamed("Store_Code", "Store_ID") \
    .withColumnRenamed("Branch_Name", "Store_Name") \
    .withColumnRenamed("Town", "City") \
    .withColumnRenamed("Province", "State") \
    .withColumnRenamed("Zone", "Region") \
    .withColumnRenamed("Store_Manager", "Manager") \
    .withColumnRenamed("Created_On", "Created_At") \
    .withColumnRenamed("Updated_On", "Last_Updated")

In [0]:
stores_b = stores_b.select(stores_a.columns)
silver_stores = stores_a.unionByName(stores_b)

In [0]:
# in company B, North was N and so on, found out while doing sql analysis, hence added this step
silver_stores = silver_stores.withColumn(
    "Region",
    when(col("Region") == "N", "North")
    .when(col("Region") == "S", "South")
    .when(col("Region") == "E", "East")
    .when(col("Region") == "W", "West")
    .otherwise(col("Region"))
)

In [0]:
silver_stores.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path + "stores")

In [0]:
# verifying count in final dataset
spark.read.format("delta") \
    .load("/Volumes/workspace/default/fmcg_data/silver/stores") \
    .count()

43

loading orders data

In [0]:
 orders_a = spark.read.format("delta").load(bronze_path + "orders_a")
 orders_b = spark.read.format("delta").load(bronze_path + "orders_b")

CLEANING ORDERS_A

In [0]:
orders_a.printSchema()
#display(orders_a)
print("Rows:", orders_a.count())

root
 |-- Order_ID: integer (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Customer_ID: integer (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Store_ID: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Payment_Mode: string (nullable = true)
 |-- Created_At: timestamp (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)

Rows: 10020


In [0]:
print("Missing Values:\n")

orders_a.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orders_a.columns
]).show()

Missing Values:

+--------+----------+-----------+----------+--------+--------+--------+------------+----------+------------+
|Order_ID|Order_Date|Customer_ID|Product_ID|Store_ID|Quantity|Discount|Payment_Mode|Created_At|Last_Updated|
+--------+----------+-----------+----------+--------+--------+--------+------------+----------+------------+
|       0|         0|          0|         0|       0|       0|     200|           0|         0|           0|
+--------+----------+-----------+----------+--------+--------+--------+------------+----------+------------+



In [0]:
orders_a= orders_a.fillna({"Discount": 0.0})
print("Missing Values:\n")

orders_a.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orders_a.columns
]).show()

Missing Values:

+--------+----------+-----------+----------+--------+--------+--------+------------+----------+------------+
|Order_ID|Order_Date|Customer_ID|Product_ID|Store_ID|Quantity|Discount|Payment_Mode|Created_At|Last_Updated|
+--------+----------+-----------+----------+--------+--------+--------+------------+----------+------------+
|       0|         0|          0|         0|       0|       0|       0|           0|         0|           0|
+--------+----------+-----------+----------+--------+--------+--------+------------+----------+------------+



In [0]:
orders_a.groupBy("Order_ID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

orders_a = orders_a.dropDuplicates(["Order_ID"])

+--------+-----+
|Order_ID|count|
+--------+-----+
|  503851|    2|
|  506112|    2|
|  507866|    2|
|  507782|    2|
|  506979|    2|
|  507776|    2|
|  504963|    2|
|  506812|    2|
|  501934|    2|
|  505438|    2|
|  508419|    2|
|  502042|    2|
|  509985|    2|
|  501990|    2|
|  502419|    2|
|  501218|    2|
|  507040|    2|
|  508518|    2|
|  503887|    2|
|  509954|    2|
+--------+-----+



In [0]:
# standardising
orders_a = orders_a.withColumn(
    "Payment_Mode",
    initcap(col("Payment_Mode"))
)

In [0]:
orders_a.filter(col("Quantity") <= 0).show()

+--------+-------------------+-----------+----------+--------+--------+--------+------------+-------------------+-------------------+
|Order_ID|         Order_Date|Customer_ID|Product_ID|Store_ID|Quantity|Discount|Payment_Mode|         Created_At|       Last_Updated|
+--------+-------------------+-----------+----------+--------+--------+--------+------------+-------------------+-------------------+
|  504281|2023-01-30 23:38:00|       1444|     P1064|    S006|      -1|     0.2|      Wallet|2023-01-31 00:32:00|2023-03-19 00:32:00|
|  504403|2023-02-28 10:20:00|       1295|     P1121|    S022|      -1|    0.15|        Cash|2023-02-28 12:27:00|2023-03-30 12:27:00|
|  505822|2026-03-30 06:49:00|       1478|     P1114|    S009|      -1|     0.0|         Upi|2026-03-30 07:36:00|2026-05-05 07:36:00|
|  500386|2023-06-03 17:57:00|       1305|     P1124|    S023|      -1|    0.15|      Wallet|2023-06-03 18:20:00|2023-06-24 18:20:00|
|  501865|2024-02-09 04:18:00|       1339|     P1116|    S008|

In [0]:
orders_a = orders_a.withColumn(
    "Quantity",
    abs(col("Quantity"))
)

In [0]:
print("Rows:", orders_a.count())

orders_a.groupBy("Order_ID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

#display(orders_a)

Rows: 10000
+--------+-----+
|Order_ID|count|
+--------+-----+
+--------+-----+



CLEANING ORDERS_B

In [0]:
orders_b.printSchema()

#display(orders_b)

print("Rows:", orders_b.count())

root
 |-- Txn_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Cust_ID: string (nullable = true)
 |-- Prod_ID: string (nullable = true)
 |-- Store_Code: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Payment_Mode: string (nullable = true)
 |-- Created_On: date (nullable = true)
 |-- Updated_On: date (nullable = true)

Rows: 7020


In [0]:
print("Missing Values:\n")

orders_b.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orders_b.columns
]).show()

Missing Values:

+------+----------+-------+-------+----------+--------+--------+------------+----------+----------+
|Txn_ID|Order_Date|Cust_ID|Prod_ID|Store_Code|Quantity|Discount|Payment_Mode|Created_On|Updated_On|
+------+----------+-------+-------+----------+--------+--------+------------+----------+----------+
|     0|         0|      0|      0|         0|       0|    1229|           0|         0|         0|
+------+----------+-------+-------+----------+--------+--------+------------+----------+----------+



In [0]:
orders_b.groupBy("Txn_ID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

orders_b = orders_b.dropDuplicates(["Txn_ID"])

+--------+-----+
|  Txn_ID|count|
+--------+-----+
|TB102025|    2|
|TB106407|    2|
|TB103425|    2|
|TB102399|    2|
|TB106216|    2|
|TB106027|    2|
|TB100133|    2|
|TB102945|    2|
|TB103255|    2|
|TB106363|    2|
|TB106501|    2|
|TB100764|    2|
|TB101942|    2|
|TB100264|    2|
|TB102265|    2|
|TB104280|    2|
|TB104351|    2|
|TB105590|    2|
|TB105713|    2|
|TB106749|    2|
+--------+-----+



In [0]:
orders_b= orders_b.fillna({"Discount": 0.0}) #null means no discount

In [0]:
# converting quantities to positive if there are any negative ones
orders_b = orders_b.withColumn(
    "Quantity",
    abs(col("Quantity"))
)

In [0]:
# standardize Payment Mode
orders_b = orders_b.withColumn(
    "Payment_Mode",
    initcap(col("Payment_Mode"))
)

In [0]:
# validate
print("Rows:", orders_b.count())

orders_b.groupBy("Txn_ID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

Rows: 7000
+------+-----+
|Txn_ID|count|
+------+-----+
+------+-----+



Harmonize Schema , Merge Tables and Save silver table

In [0]:
orders_b = orders_b \
    .withColumnRenamed("Txn_ID", "Order_ID") \
    .withColumnRenamed("Cust_ID", "Customer_ID") \
    .withColumnRenamed("Prod_ID", "Product_ID") \
    .withColumnRenamed("Store_Code", "Store_ID") \
    .withColumnRenamed("Created_On", "Created_At") \
    .withColumnRenamed("Updated_On", "Last_Updated")

In [0]:
# verify schema 
orders_a.printSchema()
orders_b.printSchema()


root
 |-- Order_ID: integer (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Customer_ID: integer (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Store_ID: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = false)
 |-- Payment_Mode: string (nullable = true)
 |-- Created_At: timestamp (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)

root
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Store_ID: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = false)
 |-- Payment_Mode: string (nullable = true)
 |-- Created_At: date (nullable = true)
 |-- Last_Updated: date (nullable = true)



In [0]:
# match datatypes
orders_b = orders_b \
    .withColumn("Order_Date", col("Order_Date").cast("timestamp")) \
    .withColumn("Created_At", col("Created_At").cast("timestamp")) \
    .withColumn("Last_Updated", col("Last_Updated").cast("timestamp"))\
    .withColumn("Order_ID", col("Order_ID").cast("string")) 

orders_a= orders_a\
    .withColumn("Customer_ID", col("Customer_ID").cast("string"))\
    .withColumn("Order_ID", col("Order_ID").cast("string")) 

In [0]:
# verify schema 
orders_a.printSchema()
orders_b.printSchema()


root
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Store_ID: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = false)
 |-- Payment_Mode: string (nullable = true)
 |-- Created_At: timestamp (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)

root
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Store_ID: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = false)
 |-- Payment_Mode: string (nullable = true)
 |-- Created_At: timestamp (nullable = true)
 |-- Last_Updated: timestamp (nullable = true)



In [0]:
silver_orders = orders_a.unionByName(orders_b)


In [0]:
silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path + "orders")

In [0]:
# verifying count in final dataset
spark.read.format("delta") \
    .load("/Volumes/workspace/default/fmcg_data/silver/orders") \
    .count()

17000